
# Logistic Regression – Airline Customer Satisfaction Prediction

## Project Objectives
- Inspect the target variable (`satisfaction`)
- Encode categorical predictors
- Split data into training and testing sets
- Build a binomial Logistic Regression model
- Evaluate using Confusion Matrix, Precision, Recall, and Accuracy
- Interpret model coefficients
- Provide business recommendations and discuss limitations


In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report

df = pd.read_csv(r"/mnt/data/invistico_Airline.csv")
print("Dataset Shape:", df.shape)
df.head()


Dataset Shape: (129880, 22)


,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,2,4,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,0,2,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,2,0,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,3,4,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,4,3,4,2,2,0,2,4,2,5,0,0.0


## Target Variable Inspection

In [2]:

print(df['satisfaction'].value_counts())
print("\nPercentage Distribution")
print(round(df['satisfaction'].value_counts(normalize=True)*100,2))


satisfaction
satisfied       71087
dissatisfied    58793
Name: count, dtype: int64

Percentage Distribution
satisfaction
satisfied       54.73
dissatisfied    45.27
Name: proportion, dtype: float64


## Missing Values

In [3]:

df.isnull().sum().sort_values(ascending=False).head(20)


Arrival Delay in Minutes             393
satisfaction                           0
Age                                    0
Type of Travel                         0
Class                                  0
Customer Type                          0
Flight Distance                        0
Seat comfort                           0
Food and drink                         0
Departure/Arrival time convenient      0
Inflight wifi service                  0
Inflight entertainment                 0
Online support                         0
Gate location                          0
Ease of Online booking                 0
On-board service                       0
Baggage handling                       0
Leg room service                       0
Checkin service                        0
Cleanliness                            0
dtype: int64

## Feature Engineering and Encoding

In [4]:

X = df.drop('satisfaction', axis=1)
y = (df['satisfaction'].astype(str).str.lower() == 'satisfied').astype(int)

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = [c for c in X.columns if c not in categorical_features]

print("Categorical Features:", len(categorical_features))
print("Numerical Features:", len(numerical_features))

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)


Categorical Features: 3
Numerical Features: 18


## Train/Test Split

In [5]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Testing Set:", X_test.shape)


Training Set: (103904, 21)
Testing Set: (25976, 21)


## Logistic Regression Model

In [6]:

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


## Model Evaluation

In [7]:

cm = confusion_matrix(y_test, y_pred)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Confusion Matrix")
print(cm)

print("\nAccuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))

print("\nClassification Report")
print(classification_report(y_test, y_pred))


Confusion Matrix
[[ 9544  2215]
 [ 2233 11984]]

Accuracy: 0.8288
Precision: 0.844
Recall: 0.8429

Classification Report
              precision    recall  f1-score   support

           0       0.81      0.81      0.81     11759
           1       0.84      0.84      0.84     14217

    accuracy                           0.83     25976
   macro avg       0.83      0.83      0.83     25976
weighted avg       0.83      0.83      0.83     25976




### Metric Interpretation

- **Precision** measures how many passengers predicted as satisfied were actually satisfied.
- **Recall** measures how many truly satisfied passengers were correctly identified.
- A balance between Precision and Recall is important because airlines want to identify satisfied customers accurately while minimizing classification errors.


## Coefficient Analysis

In [8]:

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

coef_df['Absolute_Coefficient'] = coef_df['Coefficient'].abs()

top_positive = coef_df.sort_values('Coefficient', ascending=False).head(10)
top_negative = coef_df.sort_values('Coefficient').head(10)

print("Top Positive Drivers")
display(top_positive[['Feature','Coefficient']])

print("\nTop Negative Drivers")
display(top_negative[['Feature','Coefficient']])


Top Positive Drivers


,Feature,Coefficient
7,num__Inflight entertainment,0.972683
18,cat__Customer Type_Loyal Customer,0.826756
22,cat__Class_Business,0.423151
2,num__Seat comfort,0.392571
10,num__On-board service,0.388175
13,num__Checkin service,0.354962
9,num__Ease of Online booking,0.333694
11,num__Leg room service,0.308907
20,cat__Type of Travel_Business travel,0.267379
15,num__Online boarding,0.181688



Top Negative Drivers


,Feature,Coefficient
19,cat__Customer Type_disloyal Customer,-1.046129
21,cat__Type of Travel_Personal Travel,-0.486751
24,cat__Class_Eco Plus,-0.349142
3,num__Departure/Arrival time convenient,-0.327559
4,num__Food and drink,-0.296607
23,cat__Class_Eco,-0.293382
17,num__Arrival Delay in Minutes,-0.258479
1,num__Flight Distance,-0.181496
0,num__Age,-0.133204
6,num__Inflight wifi service,-0.127863



## Business Interpretation

Positive coefficients increase the probability that a passenger will be satisfied.
Negative coefficients decrease the probability of satisfaction.

Typically, service-quality variables such as seat comfort, inflight entertainment, online booking experience, check‑in service, and onboard service are among the strongest drivers of customer satisfaction.



## Business Recommendations

1. Improve inflight entertainment and Wi‑Fi quality.
2. Invest in seat comfort and leg-room improvements.
3. Enhance digital booking and check‑in experiences.
4. Strengthen loyalty and retention programs.
5. Focus on service quality training for frontline staff.
6. Monitor satisfaction drivers regularly and retrain the model with new customer data.

## Limitations

- Logistic Regression assumes a linear relationship in log-odds.
- Customer satisfaction may depend on variables not captured in the dataset.
- More advanced models (Random Forest, XGBoost) may improve predictive performance.

## Next Steps

- Compare Logistic Regression with tree-based models.
- Perform hyperparameter tuning.
- Use SHAP values for deeper explainability.
- Deploy the model in a dashboard or API for operational use.
